# Оценка моделей в Google Colab
# Этот блокнот демонстрирует, как использовать инструменты metric_standart для оценки нейронных сетей

In [1]:
# Ячейка 1: Установка необходимых пакетов
!pip install tensorflow keras scikit-learn numpy pandas matplotlib seaborn chardet

In [2]:
# Ячейка 2: Клонирование репозитория или загрузка файлов
# Вариант 1: Клонирование репозитория (раскомментируйте при необходимости)
# !git clone https://your-repo-url.git
# %cd your-repo-name

In [3]:
# Вариант 2: Загрузка файлов проекта
# Запустите эту ячейку и используйте диалог загрузки файлов
from google.colab import files as colab_files
import os
import shutil
import zipfile

# Создание директорий для загрузки
!mkdir -p uploaded_files
!mkdir -p metric_standart
!mkdir -p model_save_preset/models
!mkdir -p data

# Загрузка файлов проекта
print("Пожалуйста, загрузите ZIP-архив проекта, содержащий модели и данные")
uploaded = colab_files.upload()

# Распаковка загруженного ZIP-архива
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('uploaded_files')
        print(f"Извлечено {filename} в директорию uploaded_files/")

Пожалуйста, загрузите ZIP-архив проекта, содержащий модели и данные


Saving project.zip to project.zip
Извлечено project.zip в директорию uploaded_files/


In [4]:
# Ячейка 3: Копирование необходимых файлов в соответствующие директории
# Копирование Python-файлов metric_standart
!cp uploaded_files/metric_standart/*.py metric_standart/
!mkdir -p metric_standart/plots

# Копирование моделей
!cp -r uploaded_files/model_save_preset/models/* model_save_preset/models/

# Копирование данных
!cp uploaded_files/data/data.csv data/

In [5]:
# Ячейка 4: Проверка структуры директорий
!ls -la metric_standart
!ls -la model_save_preset/models
!ls -la data

total 64
drwxr-xr-x 3 root root  4096 Apr 29 04:10 .
drwxr-xr-x 1 root root  4096 Apr 29 04:10 ..
-rw-r--r-- 1 root root 11532 Apr 29 04:10 analyze_results.py
-rw-r--r-- 1 root root 33791 Apr 29 04:10 model_evaluator.py
drwxr-xr-x 2 root root  4096 Apr 29 04:10 plots
-rw-r--r-- 1 root root  3077 Apr 29 04:10 run_evaluation.py
total 24
drwxr-xr-x 6 root root 4096 Apr 29 04:10  .
drwxr-xr-x 3 root root 4096 Apr 29 04:08  ..
drwxr-xr-x 2 root root 4096 Apr 29 04:10 '1 old'
drwxr-xr-x 2 root root 4096 Apr 29 04:10 '2 new'
drwxr-xr-x 2 root root 4096 Apr 29 04:10 '3 alt_model'
drwxr-xr-x 2 root root 4096 Apr 29 04:10 '4 alt_new_model'
total 416
drwxr-xr-x 2 root root   4096 Apr 29 04:10 .
drwxr-xr-x 1 root root   4096 Apr 29 04:10 ..
-rw-r--r-- 1 root root 416090 Apr 29 04:10 data.csv


In [6]:
# Ячейка 5: Исправление путей импорта в модуле metric_standart
%%writefile metric_standart/__init__.py
"""
Инструменты для оценки моделей

Этот пакет предоставляет утилиты для оценки моделей нейронных сетей
с дополнительными метриками и визуализацией их производительности.
"""

from metric_standart.model_evaluator import (
    load_and_prepare_data,
    load_models_from_directory,
    evaluate_models,
    save_evaluation_results,
    plot_prediction_comparison,
    save_metrics_to_json
)

__all__ = [
    'load_and_prepare_data',
    'load_models_from_directory',
    'evaluate_models',
    'save_evaluation_results',
    'plot_prediction_comparison',
    'save_metrics_to_json'
]

Writing metric_standart/__init__.py


In [7]:
# Ячейка 6: Импорт модулей оценки
import sys
sys.path.append('.')

from metric_standart.model_evaluator import (
    load_and_prepare_data,
    load_models_from_directory,
    evaluate_models,
    save_evaluation_results,
    plot_prediction_comparison,
    save_metrics_to_json
)

In [8]:
# Ячейка 7: Загрузка и подготовка данных
data_path = 'data/data.csv'
print(f"Загрузка и подготовка данных из {data_path}...")

try:
    data = load_and_prepare_data(data_path, time_step=5)
    print("Данные успешно загружены!")
except Exception as e:
    print(f"Ошибка загрузки данных: {e}")

Загрузка и подготовка данных из data/data.csv...
Данные успешно загружены!


/content/metric_standart/model_evaluator.py:78: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].mean(), inplace=True)


In [9]:
# Ячейка 8: Загрузка моделей
models_dir = 'model_save_preset/models'
print(f"Загрузка моделей из {models_dir}...")

models = load_models_from_directory(models_dir)

# Подсчет загруженных моделей
model_count = sum(len(group_models) for group_models in models.values())
print(f"Загружено {model_count} моделей из {len(models)} групп")


Загрузка моделей из model_save_preset/models...
Найдено 4 групп моделей: 1 old, 2 new, 3 alt_model, 4 alt_new_model

Обработка группы: 1 old (путь: model_save_preset/models/1 old)
  Найдено 5 моделей: rnn.h5, bidirectional.h5, gru.h5, deep_rnn.h5, lstm.h5
  Загрузка модели rnn...
    ⚠ Ошибка при стандартной загрузке: Unrecognized keyword arguments passed to SimpleRNN: {'time_major': False}
    ⚠ Ошибка при загрузке без компиляции: Unrecognized keyword arguments passed to SimpleRNN: {'time_major': False}
    ⚠ Ошибка при загрузке с расширенными объектами: module 'keras._tf_keras.keras.metrics' has no attribute 'mean_absolute_error'
    Попытка загрузки старой модели из группы 1 old специальным способом...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 50)             │         2,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,751 (10.75 KB)

...

    ✓ Успешно загружена модель model_cnn_lstm_20250407_1039
  Загрузка модели model_cnn_lstm_best_20250407_1041...


    ✓ Успешно загружена модель model_cnn_lstm_best_20250407_1041
  Загрузка модели model_ensemble_20250407_1041...


    ✓ Успешно загружена модель model_ensemble_20250407_1041
  Загрузка модели model_gru_20250407_1030...


    ✓ Успешно загружена модель model_gru_20250407_1030
  Загрузка модели model_lstm_20250407_1028...
    ✓ Успешно загружена модель model_lstm_20250407_1028
  Загрузка модели model_bidirectional_20250407_1036...


    ✓ Успешно загружена модель model_bidirectional_20250407_1036

Итоги загрузки моделей:
  Всего найдено: 21 моделей
  Успешно загружено: 21 моделей
  Не удалось загрузить: 0 моделей
Загружено 21 моделей из 4 групп


In [10]:
# Ячейка 8.5: Проверка моделей на наличие проблем
print("Проверка загруженных моделей на наличие проблем...")

if not models or sum(len(group_models) for group_models in models.values()) == 0:
    print("Предупреждение: Не удалось загрузить ни одной модели. Возможные причины:")
    print("1. Неверная структура каталогов")
    print("2. Отсутствие файлов моделей (.h5)")
    print("3. Несовместимость версий TensorFlow/Keras")

    # Проверка структуры каталогов
    !find model_save_preset -type d | sort

    # Проверка наличия файлов моделей
    !find model_save_preset -name "*.h5" | wc -l

    print("\nИнформация о версиях библиотек:")
    !python -c "import tensorflow as tf; print(f'TensorFlow version: {tf.__version__}')"
    !python -c "import keras; print(f'Keras version: {keras.__version__}')"

    print("\nПытаемся загрузить модели напрямую...")

    import tensorflow as tf

    # Обновляем custom_objects для поддержки дополнительных слоев и функций потерь
    custom_objects = {
        'mse': tf.keras.losses.MeanSquaredError(),
        'mae': tf.keras.losses.MeanAbsoluteError(),
        'mean_squared_error': tf.keras.losses.MeanSquaredError(),
        'mean_absolute_error': tf.keras.losses.MeanAbsoluteError(),
        'mape': tf.keras.losses.MeanAbsolutePercentageError(),
        'mean_absolute_percentage_error': tf.keras.losses.MeanAbsolutePercentageError()
    }

    # Попытка загрузить модели напрямую
    import glob
    model_files = glob.glob('model_save_preset/models/**/*.h5', recursive=True)

    if model_files:
        print(f"Найдено {len(model_files)} файлов моделей. Пробуем загрузить первую модель напрямую...")
        try:
            model = tf.keras.models.load_model(model_files[0], custom_objects=custom_objects)
            print(f"Модель {model_files[0]} успешно загружена напрямую! Структура модели:")
            model.summary()
        except Exception as e:
            print(f"Ошибка при загрузке модели напрямую: {e}")
            print("Рекомендации:")
            print("1. Обновите TensorFlow до версии, совместимой с вашими моделями")
            print("2. Если модели были сохранены с кастомными слоями или функциями потерь, добавьте их в custom_objects")
    else:
        print("Не найдено ни одного файла модели (.h5) в каталоге model_save_preset/models/")
else:
    print(f"Загружено {sum(len(group_models) for group_models in models.values())} моделей из {len(models)} групп. Все в порядке!")

Проверка загруженных моделей на наличие проблем...
Загружено 21 моделей из 4 групп. Все в порядке!


In [11]:
# Ячейка 9: Оценка моделей
print("Оценка моделей с дополнительными метриками...")
results = evaluate_models(models, data)

Оценка моделей с дополнительными метриками...
Input shape for evaluation: (871, 5, 3)
Created alternative input with 2 features: (871, 5, 2)
Created version with 10 time steps: (871, 10, 3)
Created 10-timestep version with 2 features: (871, 10, 2)

Группа: 1 old
  Оценка модели: rnn
    ⚠ Warning: MAPE calculation skipped due to zero values in test data
    ✓ MSE: 0.415047, MAE: 0.595991, R²: -7.110695
  Оценка модели: bidirectional
    ⚠ Warning: MAPE calculation skipped due to zero values in test data
    ✓ MSE: 0.444108, MAE: 0.626741, R²: -7.678581
  Оценка модели: gru
    ⚠ Warning: MAPE calculation skipped due to zero values in test data
    ✓ MSE: 0.276333, MAE: 0.477181, R²: -4.399996
  Оценка модели: deep_rnn
    ⚠ Warning: MAPE calculation skipped due to zero values in test data
    ✓ MSE: 0.195556, MAE: 0.392374, R²: -2.821475
  Оценка модели: lstm
    ⚠ Warning: MAPE calculation skipped due to zero values in test data
    ✓ MSE: 0.363504, MAE: 0.558699, R²: -6.103448

Групп

In [12]:
# Ячейка 10: Сохранение результатов в CSV
output_file = 'metric_standart/extended_model_metrics.csv'
save_evaluation_results(results, output_file)
print(f"Результаты оценки сохранены в {output_file}")

Результаты оценки сохранены в metric_standart/extended_model_metrics.csv (записано 21 моделей)
Результаты оценки сохранены в metric_standart/extended_model_metrics.csv


In [13]:
# Ячейка 10.5: Сохранение метрик в JSON файлы
print("Сохранение метрик в JSON файлах...")

# Создаем директорию для истории метрик
history_dir = 'model_save_preset/history'
!mkdir -p "{history_dir}"

# Удаляем существующие файлы перед созданием новых
!find "{history_dir}" -name "*.json" -delete
print("Существующие JSON файлы удалены")

# Сохраняем метрики с помощью функции save_metrics_to_json
save_metrics_to_json(results, history_dir)

# Проверка результатов сохранения
print("\nСписок всех сохраненных .json файлов:")
!find "{history_dir}" -name "*.json" | sort

total_found = !find "{history_dir}" -name "*.json" | wc -l
total_expected = sum(len(results.get(group, {})) for group in results.keys())
print(f"Найдено {total_found[0]} .json файлов из {total_expected} ожидаемых")

# Дополнительная проверка - вывод содержимого случайного .json файла для подтверждения корректности
json_files = !find "{history_dir}" -name "*.json"
if json_files:
    sample_file = json_files[0]
    print(f"\nПроверка содержимого файла {sample_file}:")
    try:
        import json
        with open(sample_file, 'r', encoding='utf-8') as f:
            sample_data = json.load(f)
        print(f"Группа: {sample_data.get('group_name')}")
        print(f"Модель: {sample_data.get('model_name')}")
        metrics = sample_data.get('history', {})
        print(f"Метрики: {list(metrics.keys())}")
    except Exception as e:
        print(f"Ошибка чтения файла: {str(e)}")

Сохранение метрик в JSON файлах...
Существующие JSON файлы удалены
Создана/проверена директория группы: model_save_preset/history/1 old
Успешно: метрики для 'rnn' из группы '1 old' сохранены в model_save_preset/history/1 old/rnn.json
Успешно: метрики для 'bidirectional' из группы '1 old' сохранены в model_save_preset/history/1 old/bidirectional.json
Успешно: метрики для 'gru' из группы '1 old' сохранены в model_save_preset/history/1 old/gru.json
Успешно: метрики для 'deep_rnn' из группы '1 old' сохранены в model_save_preset/history/1 old/deep_rnn.json
Успешно: метрики для 'lstm' из группы '1 old' сохранены в model_save_preset/history/1 old/lstm.json
Создана/проверена директория группы: model_save_preset/history/2 new
Успешно: метрики для 'hybrid_model_three' из группы '2 new' сохранены в model_save_preset/history/2 new/hybrid_model_three.json
Успешно: метрики для 'hybrid_model_five' из группы '2 new' сохранены в model_save_preset/history/2 new/hybrid_model_five.json
Успешно: метрики дл

In [14]:
# Ячейка 11: Генерация графиков прогнозов для каждой модели
print("Генерация графиков прогнозов для моделей...")
plots_dir = 'metric_standart/plots'
!mkdir -p "{plots_dir}"

# Ограничиваем количество моделей для генерации графиков, чтобы не создавать слишком много файлов
max_models_per_group = 2
model_count = 0

for group_name, group_models in models.items():
    print(f"Группа: {group_name}")
    counter = 0

    for model_name in list(group_models.keys())[:max_models_per_group]:
        try:
            print(f"  Создание графика для модели {model_name}...")
            plot_prediction_comparison(
                models, data, group_name, model_name,
                num_samples=100, output_dir=plots_dir
            )
            counter += 1
            model_count += 1
        except Exception as e:
            print(f"  Ошибка при создании графика для {model_name}: {e}")

    print(f"  Создано {counter} графиков для группы {group_name}")

print(f"Всего создано {model_count} графиков прогнозов")

Генерация графиков прогнозов для моделей...
Группа: 1 old
  Создание графика для модели rnn...
Prediction comparison plot saved to metric_standart/plots/1 old_rnn_prediction.png
  Создание графика для модели bidirectional...
Prediction comparison plot saved to metric_standart/plots/1 old_bidirectional_prediction.png
  Создано 2 графиков для группы 1 old
Группа: 2 new
  Создание графика для модели hybrid_model_three...
Prediction comparison plot saved to metric_standart/plots/2 new_hybrid_model_three_prediction.png
  Создание графика для модели hybrid_model_five...
Prediction comparison plot saved to metric_standart/plots/2 new_hybrid_model_five_prediction.png
  Создано 2 графиков для группы 2 new
Группа: 3 alt_model
  Создание графика для модели model_gru_20250407_0743...
  Ошибка при создании графика для model_gru_20250407_0743: Graph execution error:

Detected at node sequential_1_1/gru_1/while/gru_cell_1/MatMul defined at (most recent call last):
  File "<frozen runpy>", line 198, i

In [15]:
# Ячейка 12: Анализ - Импорт модуля анализа
from metric_standart.analyze_results import (
    load_metrics,
    create_comparison_table,
    plot_metric_comparison,
    plot_metrics_radar,
    plot_group_performance,
    save_summary_report
)

In [16]:
# Ячейка 13: Загрузка и анализ результатов
try:
    df = load_metrics(output_file)
    print(f"Загружены метрики для {len(df)} моделей")

    # Проверка, что датафрейм содержит данные
    if len(df) == 0:
        print("Предупреждение: В файле метрик нет данных. Проверьте результаты оценки моделей.")
    else:
        # Создание таблицы сравнения
        comparison = create_comparison_table(df)
        print("\nТаблица сравнения моделей:")
        display(comparison)
except Exception as e:
    print(f"Ошибка при обработке файла метрик: {e}")
    print("Проверьте, что файл метрик был правильно создан и содержит необходимые данные.")

    # Создадим пустой датафрейм для дальнейшего использования, чтобы избежать ошибок
    import pandas as pd
    df = pd.DataFrame(columns=['Group', 'Model'])

Загружены метрики для 21 моделей

Таблица сравнения моделей:


mae       mse  \
Group           Model                                                        
1 old           bidirectional                           0.626741  0.444108   
                deep_rnn                                0.392374  0.195556   
                gru                                     0.477181  0.276333   
                lstm                                    0.558699  0.363504   
                rnn                                     0.595991  0.415047   
2 new           hybrid_model                            0.190843  0.052333   
                hybrid_model_five                       0.188657  0.051943   
                hybrid_model_four                       0.226575  0.064745   
                hybrid_model_three                      0.188747  0.051940   
                hybrid_model_two                        0.188054  0.051842   
3 alt_model     model_bidirectional_20250407_0745       0.637702  0.561418   
                model_gru_20250407_0743                 0.463595  0.321451   
                model_gru_best_20250407_0745            0.463595  0.321451   
                model_lstm_20250407_0741                0.541274  0.489775   
4 alt_new_model model_bidirectional_20250407_1036       0.362454  0.186030   
                model_bidirectional_best_20250407_1036  0.362454  0.186030   
                model_cnn_lstm_20250407_1039            0.299395  0.115757   
                model_cnn_lstm_best_20250407_1041       0.299395  0.115757   
                model_ensemble_20250407_1041            0.639402  0.672394   
                model_gru_20250407_1030                 0.385780  0.209496   
                model_lstm_20250407_1028                0.510370  0.340918   

                                                            rmse  
Group           Model                                             
1 old           bidirectional                           0.666414  
                deep_rnn                                0.442217  
                gru                                     0.525674  
                lstm                                    0.602913  
                rnn                                     0.644242  
2 new           hybrid_model                            0.228763  
                hybrid_model_five                       0.227911  
                hybrid_model_four                       0.254451  
                hybrid_model_three                      0.227904  
                hybrid_model_two                        0.227689  
3 alt_model     model_bidirectional_20250407_0745       0.749278  
                model_gru_20250407_0743                 0.566967  
                model_gru_best_20250407_0745            0.566967  
                model_lstm_20250407_0741                0.699839  
4 alt_new_model model_bidirectional_20250407_1036       0.431312  
                model_bidirectional_best_20250407_1036  0.431312  
                model_cnn_lstm_20250407_1039            0.340230  
                model_cnn_lstm_best_20250407_1041       0.340230  
                model_ensemble_20250407_1041            0.819996  
                model_gru_20250407_1030                 0.457708  
                model_lstm_20250407_1028                0.583882

In [17]:
# Ячейка 14: Создание графиков сравнения метрик
try:
    if len(df) > 0:
        metrics_to_plot = [col for col in df.columns if col not in ['Group', 'Model']]
        if metrics_to_plot:
            plot_metric_comparison(df, metrics_to_plot, plots_dir)
        else:
            print("Нет доступных метрик для создания графиков сравнения")
    else:
        print("Нет данных для создания графиков сравнения метрик")
except Exception as e:
    print(f"Ошибка при создании графиков сравнения метрик: {e}")

Saved comparison plot for mse to metric_standart/plots/comparison_mse.png
Saved comparison plot for mae to metric_standart/plots/comparison_mae.png
Saved comparison plot for rmse to metric_standart/plots/comparison_rmse.png
Saved comparison plot for r2 to metric_standart/plots/comparison_r2.png
Saved comparison plot for explained_var to metric_standart/plots/comparison_explained_var.png
Saved comparison plot for max_error to metric_standart/plots/comparison_max_error.png
Saved comparison plot for median_abs_error to metric_standart/plots/comparison_median_abs_error.png
Saved comparison plot for mape to metric_standart/plots/comparison_mape.png


In [18]:
# Ячейка 15: Создание радарной диаграммы, сравнивающей лучшие модели
try:
    if len(df) > 0 and len([col for col in df.columns if col not in ['Group', 'Model']]) > 0:
        plot_metrics_radar(df, plots_dir)
    else:
        print("Недостаточно данных для создания радарной диаграммы")
except Exception as e:
    print(f"Ошибка при создании радарной диаграммы: {e}")

Saved radar chart to metric_standart/plots/radar_chart.png


In [19]:
# Ячейка 16: Создание графиков производительности по группам
try:
    if len(df) > 0:
        for metric in ['rmse', 'mae', 'r2']:
            if metric in df.columns:
                plot_group_performance(df, metric, plots_dir)

        # Если ни одна из стандартных метрик не найдена, попробуем использовать любую доступную
        metrics_found = any(metric in df.columns for metric in ['rmse', 'mae', 'r2'])
        if not metrics_found:
            available_metrics = [col for col in df.columns if col not in ['Group', 'Model']]
            if available_metrics:
                print(f"Стандартные метрики не найдены, использую доступную метрику: {available_metrics[0]}")
                plot_group_performance(df, available_metrics[0], plots_dir)
            else:
                print("Нет доступных метрик для создания графиков по группам")
    else:
        print("Нет данных для создания графиков производительности по группам")
except Exception as e:
    print(f"Ошибка при создании графиков производительности по группам: {e}")

Saved group comparison plot for rmse to metric_standart/plots/group_comparison_rmse.png
Saved group comparison plot for mae to metric_standart/plots/group_comparison_mae.png
Saved group comparison plot for r2 to metric_standart/plots/group_comparison_r2.png


In [20]:
# Ячейка 17: Создание отчета с результатами
try:
    if len(df) > 0 and len([col for col in df.columns if col not in ['Group', 'Model']]) > 0:
        summary_file = 'metric_standart/model_summary.txt'
        save_summary_report(df, summary_file)

        # Вывод отчета с результатами
        print("\nСводка оценки моделей:")
        with open(summary_file, 'r') as f:
            summary_text = f.read()
        print(summary_text)
    else:
        print("Недостаточно данных для создания отчета с результатами")
except Exception as e:
    print(f"Ошибка при создании отчета с результатами: {e}")

Ошибка при создании отчета с результатами: nan


/content/metric_standart/analyze_results.py:215: FutureWarning: The behavior of Series.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  best_idx = df[metric].idxmin()


In [21]:
# Ячейка 18: Скачивание результатов
# Запустите эту ячейку для скачивания результатов оценки
from google.colab import files as colab_files
import os
import glob
import shutil

# Временная директория для копирования файлов без пробелов в путях
temp_dir = 'temp_for_zip'
!mkdir -p {temp_dir}

# Функция для копирования файлов во временную директорию с переименованием
def prepare_files_for_zip(source_paths, temp_dir):
    copied_files = []
    for i, path in enumerate(source_paths):
        # Создаем понятное имя файла без пробелов в пути
        basename = os.path.basename(path)
        dirname = os.path.dirname(path).replace('/', '_').replace(' ', '_')
        new_name = f"{dirname}_{basename}"
        dest_path = os.path.join(temp_dir, new_name)

        # Копируем файл во временную директорию
        try:
            shutil.copy2(path, dest_path)
            copied_files.append(dest_path)
        except Exception as e:
            print(f"Ошибка при копировании {path}: {e}")

    return copied_files

# Проверка наличия результатов
files_to_copy = []

# Метрики и отчеты
if os.path.exists('metric_standart/extended_model_metrics.csv'):
    files_to_copy.append('metric_standart/extended_model_metrics.csv')

if os.path.exists('metric_standart/model_summary.txt'):
    files_to_copy.append('metric_standart/model_summary.txt')

# .json файлы с метриками
# Поиск обычных .json файлов
json_files = glob.glob('model_save_preset/history/**/*.json', recursive=True)
print(f"Найдено {len(json_files)} обычных .json файлов с метриками")

# Также проверяем альтернативные безопасные директории (с заменой пробелов на подчеркивания)
safe_model_groups = []
for group in models.keys():
    safe_group = group.replace(" ", "_")
    if safe_group != group:
        safe_model_groups.append(safe_group)

# Ищем дополнительные .json файлы в безопасных директориях
alt_json_files = []
for safe_group in safe_model_groups:
    safe_dir = os.path.join('model_save_preset/history', safe_group)
    if os.path.exists(safe_dir):
        alt_files = glob.glob(f'{safe_dir}/*.json')
        alt_json_files.extend(alt_files)

print(f"Дополнительно найдено {len(alt_json_files)} .json файлов в безопасных директориях")

# Объединяем все файлы
all_json_files = json_files + alt_json_files
print(f"Всего найдено {len(all_json_files)} .json файлов")

for json_file in all_json_files[:5]:  # Показываем первые 5 файлов для примера
    print(f"  - {json_file}")
if len(all_json_files) > 5:
    print(f"  - ... и еще {len(all_json_files) - 5} файлов")

files_to_copy.extend(all_json_files)

# Копируем файлы во временную директорию
if files_to_copy:
    print(f"Подготовка {len(files_to_copy)} файлов для архивации...")
    copied_files = prepare_files_for_zip(files_to_copy, temp_dir)

    # Создание архива
    if copied_files:
        # Переходим во временную директорию для создания архива
        !cd {temp_dir} && zip -r ../model_evaluation_results.zip *

        # Скачивание архива
        colab_files.download('model_evaluation_results.zip')
        print("Начата загрузка файла model_evaluation_results.zip")

        # Удаляем временную директорию
        !rm -rf {temp_dir}
    else:
        print("Ошибка при подготовке файлов для архивации")
else:
    print("Нет результатов для скачивания. Проверьте, были ли успешно созданы файлы метрик.")

Найдено 21 обычных .json файлов с метриками
Дополнительно найдено 0 .json файлов в безопасных директориях
Всего найдено 21 .json файлов
  - model_save_preset/history/2 new/hybrid_model_three.json
  - model_save_preset/history/2 new/hybrid_model.json
  - model_save_preset/history/2 new/hybrid_model_four.json
  - model_save_preset/history/2 new/hybrid_model_five.json
  - model_save_preset/history/2 new/hybrid_model_two.json
  - ... и еще 16 файлов
Подготовка 22 файлов для архивации...
  adding: metric_standart_extended_model_metrics.csv (deflated 58%)
  adding: model_save_preset_history_1_old_bidirectional.json (deflated 38%)
  adding: model_save_preset_history_1_old_deep_rnn.json (deflated 39%)
  adding: model_save_preset_history_1_old_gru.json (deflated 39%)
  adding: model_save_preset_history_1_old_lstm.json (deflated 38%)
  adding: model_save_preset_history_1_old_rnn.json (deflated 38%)
  adding: model_save_preset_history_2_new_hybrid_model_five.json (deflated 39%)
  adding: model_sa

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Начата загрузка файла model_evaluation_results.zip
